**Topic:** Tool Calling & Binding in LangChain

---

### 1. Introduction: The Problem with LLMs

Large Language Models (LLMs) have two main powers:
- **Reasoning:** They can understand and break down a question.
- **Output Generation:** They can generate text based on their parametric knowledge.
<img src="Screenshot 2026-09-05 at 7.34.51 PM.png">
**The Big Problem:** LLMs cannot **perform actions**. They are like a person who can think and speak but has no hands or feet.
- They can't access a database to make changes.
- They can't post on social media.
- They can't call an external API to get the current weather.

**The Solution:** To make LLMs truly powerful, they need "hands and feet." This is achieved by giving them **Tools**.

**What is a Tool?**
- A Tool is essentially a Python function with a specific capability.
- It is designed to interact with an LLM.
- A tool can be a built-in LangChain tool (like DuckDuckGo search) or a custom tool you create.

---

### 2. Tool Binding
<img src="Screenshot 2026-09-05 at 7.36.43 PM.png">


**Explanation:**
Tool Binding is the process of connecting a tool (or multiple tools) to an LLM. When you bind a tool, the LLM learns:
- **What tools are available:** A list of tools it can use.
- **What each tool does:** From the tool's description.
- **What input format the tool expects:** The tool's argument schema.

This is the first step. You create the tool and then "bind" it to the LLM so the LLM knows it exists and how to use it.


#### 1. Create a Tool

In [1]:

from langchain_core.tools import BaseTool
from typing import Type,Annotated
from pydantic import BaseModel,Field

# 1.Define a input schema (Pydantic Model)
class MultiplyInput(BaseModel):
    a:Annotated[int,Field(...,description="The first number to add")]
    b:Annotated[int,Field(...,description="The second number to add")]

# 2. Create a class that inherits from BaseTool
class MultiplyTool(BaseTool):
    name:str="multiply"
    description:str="Multiply two numbers"
    args_schema:Type[BaseModel]=MultiplyInput #This enforces the schema using pydantic model

    #3 Implement the _run method
    def _run(self,a:int,b:int)->int:
        """Synchronous method to multiply two numbers"""
        return a*b
    
    # Optional: Implement an async version
    # async def _arun(self, a: int, b: int) -> int:
    #     """Async version."""
    #     return a * b

# Use the Tool
my_tool=MultiplyTool()
result=my_tool.invoke({"a":2,"b":3})
print(result) #Ouput:6

# Check the tool's properties
print(my_tool.name)  # Output: multiply
print(my_tool.description)  # Output: Given two numbers a and b...
print(my_tool.args)  # Output: {'a': {'title': 'A', 'type': 'integer'}, ...}


6
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'title': 'B', 'type': 'integer'}}


####  2. Bind the tool to an LLM

In [2]:
! #pip install sentencepiece

In [3]:
## ------ Fetching the model ---------------
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os

## I tried mistral ai 7b v0.2 ,it works, but it is not good

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# Create a Chat Model instance
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=api_key)


# -------- Binding process -------

model_with_tools=model.bind_tools([MultiplyTool()]) # This is the binding step ,add tools in that list

# model_with_tools is now an LLM that knows about the 'multiply' tool.

**Important Note:** Not all LLMs support tool binding. The model you use must have this capability (e.g., many OpenAI, Anthropic, and open-source models fine-tuned for function calling).

### 3. Tool Calling
<img src="Screenshot 2026-09-05 at 7.51.30 PM.png">



**Explanation:**
Tool Calling is the process where the LLM, during a conversation, decides it needs to use a tool. It doesn't *execute* the tool itself. Instead, it **suggests** which tool to use and what arguments to pass to it.

The LLM generates a **structured output** containing:
- `name`: The name of the tool to use.
- `args`: A dictionary of arguments to pass to the tool.

This is a crucial point: **The LLM does not run the tool. It only makes a suggestion.**

In [4]:
# Using the 'model_with_tools' from the previous step
# 1. Ask a question that doesn't require a tool
response1=model_with_tools.invoke("hi, how are you?")
print(response1.content)

[{'type': 'text', 'text': "Hello! I'm doing well, thank you for asking. How can I help you today?", 'extras': {'signature': 'El4KXAERTTIPrZ4qGHhpvPFRPJvNKN8vt4MhG9xsA/kabsjmUuA1xeAKr6jdVEHOlS8COYEoHkoDPhiSa/oG1SlXoFEwm7GQHu/p8riS21kAqb1s+usgFp8PKp+KSwoW'}}]


In [5]:
# 2. Ask a question that DOES require the tool
response2=model_with_tools.invoke("can u multiply 3 by 10? ")
print(response2)
print(response2.content) #This will be empty or none
print(response2.tool_calls) #returns list of tools


content=[] additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 3, "b": 10}'}, '__gemini_function_call_thought_signatures__': {'fa22aa35-a08b-4384-afb6-8b25afbc5648': 'El4KXAERTTIPH3cfo3+rD4t7BfSYeLnfhTNpaECoVAo6XCFgfaCN7ij6cHK1niBp+6i6wsxHcHyLEeKkSfONT3fOOqflmv9K4PAOnqCrjpRE01P5/xQdgTWyqr9wHgH4'}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a07276-1eba-7683-a4e6-8a2af95be2f4-0' tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'fa22aa35-a08b-4384-afb6-8b25afbc5648', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 82, 'output_tokens': 17, 'total_tokens': 99, 'input_token_details': {'cache_read': 0}}
[]
[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'fa22aa35-a08b-4384-afb6-8b25afbc5648', 'type': 'tool_call'}]


In [6]:
print(response2.tool_calls[0])# first tool

{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'fa22aa35-a08b-4384-afb6-8b25afbc5648', 'type': 'tool_call'}


**Key Takeaway:** The LLM suggests using the `multiply` tool with `a=3` and `b=10`.


### 4. Tool Execution
<img src="Screenshot 2026-09-05 at 9.08.05 PM.png">


**Explanation:**
Tool Execution is the step where **you, as the programmer**, take the suggestion from the LLM (the `tool_calls`) and actually execute the corresponding Python function.

You extract the arguments, call the tool, and get the result. This result is wrapped in a special type of message called a `ToolMessage`.

In [7]:
# Continuing from the previous step

from langchain_core.messages import ToolMessage

# 1.Get the tool call suggestion from the LLMs Response
tool_call=response2.tool_calls[0] #get the first tool call in the list

# 2.Execute the tool call
tool_result=my_tool.invoke(tool_call) # pass the suggested response to the tool
print(f"Tool execution result: {tool_result}")  # Output: Tool execution result: 30



Tool execution result: content='30' name='multiply' tool_call_id='fa22aa35-a08b-4384-afb6-8b25afbc5648'


In [8]:
tool_result

ToolMessage(content='30', name='multiply', tool_call_id='fa22aa35-a08b-4384-afb6-8b25afbc5648')

### 5. The Full Flow with Conversation History (Important)

To get a final answer back to the user, you must maintain a conversation history (messages list) that includes the human question, the AI's tool call suggestion, and the resulting tool message. This gives the LLM the full context to generate a final, natural language answer.

In [9]:
from langchain_core.messages import HumanMessage, AIMessage

# 1. Initial Message List
messages = [HumanMessage(content="can u multiply 3 by 10?")]

# 2. Invoke the  LLM (with tools defined above) and get the AI's response
ai_response = model_with_tools.invoke(messages)

# 3. Append the AI's response (which contains the tool call) to the history
messages.append(ai_response)

# 4. Execute the tool based on the AI's suggestion
tool_call = ai_response.tool_calls[0]
tool_result = my_tool.invoke(tool_call)

# 5. Append the tool message to the history
messages.append(tool_result)

# 6. Invoke the LLM again with the full history
final_response = model_with_tools.invoke(messages)
print(final_response.content)

[{'type': 'text', 'text': '3 multiplied by 10 is 30.', 'extras': {'signature': 'El4KXAERTTIPiylgSZd7VEtbQdYAV2jCGlLDCtPDMWiFNd0SZkyjI5nA5yn0NSBpokfW2k7NdgimTonFZGLNM1KeeQXdSf4IJklFQx3n7pQoPtZNCS8shDpvQIHv8CNV'}}]


In [10]:
final_response

AIMessage(content=[{'type': 'text', 'text': '3 multiplied by 10 is 30.', 'extras': {'signature': 'El4KXAERTTIPiylgSZd7VEtbQdYAV2jCGlLDCtPDMWiFNd0SZkyjI5nA5yn0NSBpokfW2k7NdgimTonFZGLNM1KeeQXdSf4IJklFQx3n7pQoPtZNCS8shDpvQIHv8CNV'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a07276-2556-75a2-8600-2510eff863fa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 109, 'output_tokens': 11, 'total_tokens': 120, 'input_token_details': {'cache_read': 0}})

### 6. Real-World Application: Currency Converter

**Explanation:**
This example builds a currency converter that gets a *real-time* conversion rate from an external API. 
<img src="Screenshot 2026-09-05 at 9.28.26 PM.png">
It uses two tools:
1.  `get_conversion_factor`: Fetches the current exchange rate between two currencies.
2.  `convert`: Multiplies an amount by a conversion factor.

**Key Concept: Injected Tool Arguments**
A challenge arises when the LLM tries to call the `convert` tool in the same step as the `get_conversion_factor` tool. It doesn't know the real-time rate yet, so it might use an old, incorrect one from its training data. To fix this, we mark the `conversion_rate` argument in the `convert` tool as an `InjectedToolArg`. This tells the LLM not to provide that argument; it will be provided by the programmer after executing the first tool.

https://www.exchangerate-api.com/docs/pair-conversion-requests

In [11]:
#---------- Import necessary libraries----------------
import requests
import json
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import InjectedToolArg
from typing import Annotated

import os
from dotenv import load_dotenv

load_dotenv()
exchange_rate_api_key=os.getenv("exchange_rate_api")
model_api_key=os.getenv("GOOGLE_API_KEY")

#------1. Create Tools -------------------
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """Fetches the real-time currency conversion factor between a base currency and a target currency."""
    url = f"https://v6.exchangerate-api.com/v6/{exchange_rate_api_key}/pair/{base_currency}/{target_currency}"

    response=requests.get(url)
    return response.json() ## we get conversion rate from here

@tool
def convert(base_currency_value: float, conversion_rate:Annotated[float,InjectedToolArg]) -> float:
    """Given a base currency value and a conversion rate , calculates the target currency value based on base currency value."""

    return base_currency_value*conversion_rate



# --- 2. Bind Tools to LLM --- Tool Binding
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=model_api_key)
tools=[get_conversion_factor,convert]
model_with_tools=model.bind_tools(tools)

# --- 3. The Full Agentic-Like Flow ---
messages = [HumanMessage(content="What is the conversion factor between USD and INR? Based on that, convert 10 USD to INR.")]

# First LLM call: It decides to use both tools.
ai_response = model_with_tools.invoke(messages)
messages.append(ai_response)


# Process tool calls
for tool_call in ai_response.tool_calls:
    if tool_call["name"] == "get_conversion_factor":
        # Execute the first tool
        tool_result1 = get_conversion_factor.invoke(tool_call)
        # Store the conversion rate for later injection by fetching it from the tool_result
        conversion_rate=json.loads(tool_result1.content)["conversion_rate"] #changing to dictionary to access conversion_rate key
        print(f"Conversion rate fetched from api: {conversion_rate}")
        messages.append(tool_result1)
        
    if tool_call["name"]=="convert":
        # For the second tool, get its arguments
        # Inject the conversion_rate we got from the first tool
        tool_call["args"]["conversion_rate"]=conversion_rate
        # Execute the tool with the injected argument
        tool_result2 = convert.invoke(tool_call)
        messages.append(tool_result2)

# Final LLM call: It has the full history and can generate a final answer.
final_response = model_with_tools.invoke(messages)
print(final_response.content)

Conversion rate fetched from api: 94.5176
[]


We didnt get the 2nd call of tools from this model,we need a better model than this for better perofrmance..
the ideal output should look like this
**Output (Real-time, so numbers will vary):**
```
The conversion factor between USD and INR is approximately 94.5176. Based on this, 10 USD is equivalent to 945.17 INR.
```

### Please Note that Tool Calling is not Agent AI itself,but its an important part

<img src="Screenshot 2026-09-05 at 10.32.51 PM.png">

## Testing during above codes

In [14]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR? Based on that, convert 10 USD to INR.', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"base_currency": "USD", "target_currency": "INR"}'}, '__gemini_function_call_thought_signatures__': {'a5909804-9147-4c1b-bd97-49537e0eb1a3': 'El4KXAERTTIPGe/t5la29Exqma4L0NxUrA/r31f+j6MCdoXVIUlB3mmkprbfeQ97iUTIh677CKXBf1v4qgJ41IgKgtI8TLOytq/pdr3RqPqh8j//x9d9JBPhMxeAOkkm'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a07276-29c3-7ab2-9ddf-c3a265843a3a-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'INR'}, 'id': 'a5909804-9147-4c1b-bd97-49537e0eb1a3', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 164, 'output_tokens': 28, 'total_to

In [15]:
len(messages)

3

In [12]:
## for testing tool functions individually
convert.invoke(
    {
        "base_currency_value":10.0,
        "conversion_rate":94.5176
    }
)

945.176

In [13]:
## for testing tool functions
get_conversion_factor.invoke(
    {"base_currency":"USD",
    "target_currency":"INR"}
)

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1788566402,
 'time_last_update_utc': 'Sat, 05 Sep 2026 00:00:02 +0000',
 'time_next_update_unix': 1788652802,
 'time_next_update_utc': 'Sun, 06 Sep 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.5176}